In [1]:
import json
import pandas as pd
from tqdm import tqdm 
from model import *
from prompt import *

tqdm.pandas()

In [2]:
def parse_llm_response(response):
    """
    Safely parse LLM response to ensure valid JSON.
    Returns default_value if parsing fails.
    """
    if "```json" in response:
        try:
            start_idx = response.index("```json") + 7
            end_idx = response.index("```", start_idx)
            response = response[start_idx:end_idx].strip()
        except ValueError:
            pass
    elif "```" in response:
        try:
            start_idx = response.index("```") + 3
            end_idx = response.index("```", start_idx)
            response = response[start_idx:end_idx].strip()
        except ValueError:
            pass
            
    try:
        return json.loads(response)
    except json.JSONDecodeError as e:
        print(f"Failed to parse LLM response as JSON: {e}")
        print("Response was:", response[:200], "..." if len(response) > 200 else "")

### Examples

In [3]:
example1 = """
    § 585.203 With whom will BOEM consult before issuance of leases? § 585.204 What areas are available for leasing consideration? § 585.205 How will leases be mapped? § 585.206 What is the lease size? §§ 585.207-585.209 [Reserved] Competitive Lease Award Process—Pre-Auction Provisions § 585.210 What are the steps in BOEM's competitive lease award process? § 585.211 What is the Call? § 585.212 What is area identification? § 585.213 What information is included in the PSN? § 585.214 What information is included in the FSN? § 585.215 What may BOEM do to assess whether competitive interest for a lease area still exists before the auction? § 585.216 How are bidding credits awarded and used? §§ 585.217-585.219 [Reserved] Competitive Lease Award Process—Auction Provisions § 585.220 How will BOEM award leases competitively? § 585.221 What general provisions apply to all auctions? § 585.222 What other auction rules must bidders follow? § 585.223 What supplemental information will BOEM provide in a PSN and FSN? Competitive Lease Award Process—Post-Auction Provisions § 585.224 What will BOEM do after the auction? § 585.225 What happens if BOEM accepts a bid? § 585.226 What happens if the provisional winner fails to meet its obligations? §§ 585.227-585.229 [Reserved] Noncompetitive Lease Award Process § 585.230 May I request a lease if there is no Call? § 585.231 Will BOEM issue leases noncompetitively? § 585.232 May I acquire a lease noncompetitively after responding to a request for information or a Call for Information and Nominations? §§ 585.233-585.234 [Reserved] Commercial and Limited Lease Periods § 585.235 What are the lease periods for a commercial lease? § 585.236 If I have a limited lease, how long will my lease remain in effect? § 585.237 What is the effective date of a lease? § 585.238 May I develop my commercial lease in phases? § 585.239 Are there any other renewable energy research activities that will be allowed on the OCS? §§ 585.240-585.299 [Reserved] 30 CFR Part 585 (up to date as of 2/18/2025) Renewable Energy on the Outer Continental Shelf 30 CFR Part 585 (Feb. 18, 2025) 30 CFR Part 585 (Feb. 18, 2025) (enhanced display) page 2 of 98"
"""

example2 = """
    Document number: 3.
    Document title: Offshore Wind Submarine Cable Spacing Guidance
    Regulations:
    Submarine Cable Burial Depths:
    In an anchorage area or marine park: burial depths are typically around 15 ft. (5 m) below seabed.
    Otherwise, required burial depth may be as low as 3-6 ft. (1-2 m)​.
    Cable Burial Depth Requirement in New Jersey:
    New Jersey mandates a minimum burial depth of 5 ft. (1.5 m) in state waters​.
    HVAC vs. HVDC Transmission Distance:
    Wind farms close to shore (typically up to 60 to 100 km) may use HVAC in the export cables.
    For greater distances, HVDC may be the most economical transfer method​.
    Spacing of Wind Turbines:
    Generally, offshore wind turbines are spaced approximately 600 - 1000 meters apart, which influences cable routing and access for installation and maintenance​.
    Minimum Distance Between Offshore Cables:
    When two cables are separated, the minimum distance is generally determined by the footprint of seabed installation or burial equipment.
    Maximum width of burial machinery is around 10 to 12 meters, so a corridor of 30 to 50 meters between cables is recommended to avoid risk during installation​.
    Burial Depth and Separation in Germany:
    German grid operator Tennet requires parallel-laid cables to be spaced ≥100m or 3x water depth, though some cases have allowed for exceptions.
    Federal Regulation on Adverse Environmental Conditions:
    30 CFR 585.816 states that if environmental or other conditions adversely affect a cable, pipeline, or facility, a corrective action plan must be submitted within 30 days of discovery, and remedial action must be reported to BOEM within 30 days after completion​.
"""

In [ ]:
input_prompt1 = prompt.format(DOCUMENTATION = example1)
response1 = call_gpt4o_mini(input_prompt1)
parse_llm_response(response1)

In [ ]:
input_prompt2 = prompt.format(DOCUMENTATION = example2)
response2 = call_gpt4o_mini(input_prompt2)
parse_llm_response(response2)

### Scaling to the dataset

In [ ]:
df = pd.read_csv('Specs csv/1.csv')
df

In [ ]:
df['content'].iloc[1]

In [ ]:
df['prompt'] = df['content'].apply(
        lambda content: prompt.format(DOCUMENTATION=content)
    )

df

In [ ]:
df['gpt4o_mini_response'] = df['prompt'].progress_apply(call_gpt4o_mini)
df

In [ ]:
df['clean_response'] = df['gpt4o_mini_response'].progress_apply(parse_llm_response)
df

In [8]:
df.to_csv("1_response.csv",index=False)